In [1]:
import os
import pandas as pd
import json
Dir_project = '/work/desai-lab/xuanyang/Project/dataset/Nastase/narratives'
Dir_gentle = '/work/desai-lab/xuanyang/Project/dataset/Nastase/narratives/stimuli/gentle'

/home/xy6/.local/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


#### Load fMRI data needs to be run

In [2]:
# fmri Data
df_scans_filter = pd.read_csv('/work/desai-lab/xuanyang/Project/dataset/Nastase/Master_scans_252subjs.csv')
df_scans_filter

,subID,folder,files,label,task,subID_task,tag,condition,transcript,events,img
0,sub-001,fmriprep,sub-001_task-tunnel_space-MNI152NLin2009cAsym_...,mask,tunnel,sub-001-tunnel,sub-001_task-tunnel,NaN,tunnel,sub-001_task-tunnel_events.tsv,sub-001_task-tunnel_space-MNI152NLin2009cAsym_...
1,sub-001,fmriprep,sub-001_task-tunnel_desc-confounds_regressors....,confounds_json,tunnel,sub-001-tunnel,sub-001_task-tunnel_desc-confounds_regressors....,NaN,tunnel,NaN,NaN
2,sub-001,fmriprep,sub-001_task-tunnel_desc-confounds_regressors.tsv,confounds_tsv,tunnel,sub-001-tunnel,sub-001_task-tunnel_desc-confounds_regressors.tsv,NaN,tunnel,NaN,NaN
3,sub-001,afni-smooth,sub-001_task-tunnel_space-MNI152NLin2009cAsym_...,bold_clean,tunnel,sub-001-tunnel,sub-001_task-tunnel,NaN,tunnel,sub-001_task-tunnel_events.tsv,sub-001_task-tunnel_space-MNI152NLin2009cAsym_...
4,sub-001,afni-nosmooth,sub-001_task-tunnel_space-MNI152NLin2009cAsym_...,bold_clean,tunnel,sub-001-tunnel,sub-001_task-tunnel,NaN,tunnel,sub-001_task-tunnel_events.tsv,sub-001_task-tunnel_space-MNI152NLin2009cAsym_...
...,...,...,...,...,...,...,...,...,...,...,...
2485,sub-345,fmriprep,sub-345_task-notthefallintact_desc-confounds_r...,confounds_tsv,notthefallintact,sub-345-notthefallintact,sub-345_task-notthefallintact_desc-confounds_r...,intact-shortscram,notthefallintact,NaN,NaN
2486,sub-345,fmriprep,sub-345_task-notthefallintact_space-MNI152NLin...,mask,notthefallintact,sub-345-notthefallintact,sub-345_task-notthefallintact,intact-shortscram,notthefallintact,sub-345_task-notthefallintact_events.tsv,sub-345_task-notthefallintact_space-MNI152NLin...
2487,sub-345,fmriprep,sub-345_task-notthefallintact_desc-confounds_r...,confounds_json,notthefallintact,sub-345-notthefallintact,sub-345_task-notthefallintact_desc-confounds_r...,intact-shortscram,notthefallintact,NaN,NaN
2488,sub-345,afni-smooth,sub-345_task-notthefallintact_space-MNI152NLin...,bold_clean,notthefallintact,sub-345-notthefallintact,sub-345_task-notthefallintact,intact-shortscram,notthefallintact,sub-345_task-notthefallintact_events.tsv,sub-345_task-notthefallintact_space-MNI152NLin...


In [3]:
df_master_blankTR = pd.DataFrame({'transcript':df_scans_filter.transcript.unique()})
df_master_blankTR = df_master_blankTR.sort_values('transcript').reset_index(drop=True)
df_master_blankTR['blankTR'] = [0,8,8,8,2,0,0,0,3,0,8,0,3,3,2]
df_master_blankTR

,transcript,blankTR
0,21styear,0
1,black,8
2,bronx,8
3,forgot,8
4,lucy,2
5,milkywayoriginal,0
6,milkywaysynonyms,0
7,milkywayvodka,0
8,notthefallintact,3
9,pieman,0


In [4]:
import nltk
dfs_timestamps = []
for transcript in df_scans_filter.transcript.unique():
    df_timestamps = pd.read_csv(os.path.join(Dir_gentle,transcript,'align.csv'),header=None)
    df_timestamps.columns = ['Word','word_gentle','onset','offset']
    df = pd.DataFrame(nltk.pos_tag(df_timestamps['Word'].str.lower()),columns = ['Word','PoS'])
    df_timestamps['PoS'] = df['PoS']
    df_timestamps['transcript'] = transcript
    dfs_timestamps.append(df_timestamps)
dfs_timestamps = pd.concat(dfs_timestamps,ignore_index=True)

# lemmatize nltk
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
for word in dfs_timestamps['Word'].unique():
    dfs_timestamps.loc[dfs_timestamps['Word']==word,'lemma'] = lemmatizer.lemmatize(word.lower())
dfs_timestamps['word'] = dfs_timestamps['Word'].str.lower()
dfs_timestamps

,Word,word_gentle,onset,offset,PoS,transcript,lemma,word
0,Tonight,tonight,0.12,0.48,NN,tunnel,tonight,tonight
1,a,a,0.69,0.80,DT,tunnel,a,a
2,story,story,0.80,1.21,NN,tunnel,story,story
3,by,by,1.21,1.46,IN,tunnel,by,by
4,Frederik,NaN,NaN,NaN,JJ,tunnel,frederik,frederik
...,...,...,...,...,...,...,...,...
33319,tell,tell,407.41,407.61,VB,piemanpni,tell,tell
33320,me,me,407.61,407.77,PRP,piemanpni,me,me
33321,all,all,407.77,408.00,DT,piemanpni,all,all
33322,about,about,408.00,408.38,IN,piemanpni,about,about


In [12]:
import spacy
from spacy.symbols import ORTH
nlp = spacy.load('en_core_web_sm')
nlp.max_length = 1013000
# Add specials case to keep as single tokens
doc = nlp(' '.join(dfs_timestamps['Word']))
lemma_spacy = [(token.text, token.lemma_, token.pos_) for token in doc]
    

#### shift the timings to match the fmri time series

In [5]:
TR = 1.5
Dir_codes= '/work/desai-lab/xuanyang/Project/Semantic/analysis/FactorAnalysis/github/FactorAnalysis_fMRI/scripts/fMRI/timings/'
dfs_timestamps['blankTR'] = 0
for transcript in dfs_timestamps.transcript.unique():
    blankTR = df_master_blankTR.loc[df_master_blankTR.transcript==transcript,'blankTR'].values[0]
#     dfs_timestamps.loc[dfs_timestamps.transcript == transcript,['onset_addblankTR','offset_addblankTR']] = dfs_timestamps.loc[dfs_timestamps.transcript == transcript,['onset','offset']] + [blankTR*TR,blankTR*TR]
    dfs_timestamps.loc[dfs_timestamps.transcript==transcript,'blankTR'] = blankTR
dfs_timestamps['onset_addblankTR'] = dfs_timestamps['onset'] + dfs_timestamps['blankTR'] *TR
dfs_timestamps['offset_addblankTR'] = dfs_timestamps['offset'] + dfs_timestamps['blankTR'] *TR
dfs_timestamps.to_csv(os.path.join(Dir_codes,'df_timings_15transcripts.csv'),index=False)
dfs_timestamps

,Word,word_gentle,onset,offset,PoS,transcript,lemma,word,blankTR,onset_addblankTR,offset_addblankTR
0,Tonight,tonight,0.12,0.48,NN,tunnel,tonight,tonight,2,3.12,3.48
1,a,a,0.69,0.80,DT,tunnel,a,a,2,3.69,3.80
2,story,story,0.80,1.21,NN,tunnel,story,story,2,3.80,4.21
3,by,by,1.21,1.46,IN,tunnel,by,by,2,4.21,4.46
4,Frederik,NaN,NaN,NaN,JJ,tunnel,frederik,frederik,2,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
33319,tell,tell,407.41,407.61,VB,piemanpni,tell,tell,8,419.41,419.61
33320,me,me,407.61,407.77,PRP,piemanpni,me,me,8,419.61,419.77
33321,all,all,407.77,408.00,DT,piemanpni,all,all,8,419.77,420.00
33322,about,about,408.00,408.38,IN,piemanpni,about,about,8,420.00,420.38
